# Stage 04: Data Acquisition and Ingestion

Stage 04. **I based it on the lecture notebook.**

In [ ]:
# Install missing packages (uncomment and run to install).
# !pip install pandas requests python-dotenv beautifulsoup4

In [ ]:
from pathlib import Path

# Project root.
ROOT = Path.cwd()
if not (ROOT / ".env.example").exists() and (ROOT.parent / ".env.example").exists():
    ROOT = ROOT.parent

CHECKS = [
    (".env", "NEEDED", "copy .env.example to .env"),
    (".env.example", "NEEDED", "template for local secrets"),
    ("src/io_utils.py", "NEEDED", "save and validate helpers"),
]

print(f"Looking in: {ROOT}\n")
missing = 0
for rel, kind, note in CHECKS:
    here = (ROOT / rel).exists()
    if not here and kind == "NEEDED":
        missing += 1
    print(f"  [{'OK ' if here else 'MISS'}]  {kind:<8}  {rel:<34}  {note}")

if missing:
    raise FileNotFoundError(f"{missing} needed file(s) missing under {ROOT}")
print("\nAll needed files present.")

In [ ]:
import sys

import pandas as pd
import requests
from bs4 import BeautifulSoup

# Import helpers from src/.
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import get_key, load_env
from src.io_utils import save_csv, validate

# Raw outputs.
RAW = ROOT / "data" / "raw"
RAW.mkdir(parents=True, exist_ok=True)

# Load `.env`.
load_env()
print("API_KEY present:", get_key("API_KEY") is not None)

## API pull

In [ ]:
# Prismatic Evolutions on TCGCSV.
GROUP_ID = 23821
TCGCSV_BASE = f"https://tcgcsv.com/tcgplayer/3/{GROUP_ID}"
headers = {"User-Agent": "Pokemon-Set-Screener/1.0"}

# TCGCSV prices.
prices_response = requests.get(f"{TCGCSV_BASE}/prices", headers=headers, timeout=30)
prices_response.raise_for_status()
prices = pd.DataFrame(prices_response.json()["results"])

# TCGCSV product names.
products_response = requests.get(f"{TCGCSV_BASE}/products", headers=headers, timeout=30)
products_response.raise_for_status()
products = pd.DataFrame(products_response.json()["results"])

# Join prices to names.
card_prices = prices.merge(products[["productId", "name"]], on="productId", how="left")
card_prices = card_prices.rename(columns={"name": "card_name", "marketPrice": "market_price"})
card_prices["market_price"] = pd.to_numeric(card_prices["market_price"], errors="coerce")
card_prices["pulled_at"] = pd.Timestamp.now()

# Required cols, shape, NA count.
api_check = validate(card_prices, ["productId", "card_name", "market_price"])
api_check

In [ ]:
# Save API CSV.
api_csv_path = save_csv(
    card_prices,
    prefix="api",
    raw_dir=RAW,
    source="tcgcsv",
    set="prismatic-evolutions",
)

## Scrape a public table

In [ ]:
SCRAPE_URL = "https://en.wikipedia.org/wiki/List_of_Pok%C3%A9mon_Trading_Card_Game_sets"
headers = {"User-Agent": "Pokemon-Set-Screener/1.0"}

try:
    # Wikipedia TCG set list.
    wiki_response = requests.get(SCRAPE_URL, headers=headers, timeout=30)
    wiki_response.raise_for_status()
    soup = BeautifulSoup(wiki_response.text, "html.parser")
    table = soup.find("table", class_="wikitable")
    if table is None:
        raise RuntimeError("No wikitable found")

    # Header row then data rows.
    rows = []
    for table_row in table.find_all("tr"):
        cells = [cell.get_text(strip=True) for cell in table_row.find_all(["td", "th"])]
        if cells:
            rows.append(cells)
    header, *data = rows
    scraped = pd.DataFrame(data, columns=header)
except Exception as error:
    print("Scrape failed, using inline demo table:", error)
    html = "<table><tr><th>Name</th><th>Release date</th></tr><tr><td>Prismatic Evolutions</td><td>2025-01-17</td></tr></table>"
    soup = BeautifulSoup(html, "html.parser")
    rows = []
    for table_row in soup.find_all("tr"):
        cells = [cell.get_text(strip=True) for cell in table_row.find_all(["th", "td"])]
        if cells:
            rows.append(cells)
    header, *data = rows
    scraped = pd.DataFrame(data, columns=header)

# Required cols, shape, NA count.
scrape_check = validate(scraped, list(scraped.columns))
scrape_check

In [ ]:
# Save scrape CSV.
scrape_csv_path = save_csv(scraped, prefix="scrape", raw_dir=RAW, site="wikipedia", table="tcg-sets")

## Documentation

- For my API source, I use [TCGCSV](https://tcgcsv.com), which is a TCGPlayer mirror. The endpoints are ` /tcgplayer/3/23821/prices` and `/products` (Prismatic Evolutions `groupId=23821`). I didn't have to use an API key.
- For my table source I use the first `wikitable` from [List of Pokémon TCG sets](https://en.wikipedia.org/wiki/List_of_Pokémon_Trading_Card_Game_sets).
- For validation, I validate required columns, shape, total NA count via `validate()`.
- `.env` is gitignored. I kept `.env.example` in the repo.

Assumptions and risks:
- TCGCSV `/prices` is today's snapshot instead of a historical archive..
- Wikipedia can change its table layout suddenly, making the code invalid..
- Rate limits and User-Agent can still affect my code on public pages, so I may get rate-limited or IP suspended by the website if I request data very, very frequently.